In [2]:
from pathlib import Path

OUTPUT_FORMAT = "webm"
ALPHA = False

OUT_DIR = Path("media-site/animations/Math")

W = 10
H = 10
DPI = 120

FPS = 24
DURATION = 8
FRAMES = FPS * DURATION

BG = "black"

COL = "#35f6ff"

SURFACE_ALPHA = 0.28

GRID_STEP_U = 8
GRID_STEP_V = 8

GRID_GLOW_WIDTH = 1.4
GRID_CORE_WIDTH = 0.45

GRID_GLOW_ALPHA = 0.55
GRID_CORE_ALPHA = 0.95

COL_GRID_U = "white"
COL_GRID_V = "#9ffcff"

# Costa-like Surface

In [3]:
# Costa-like Surface — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/costa_like_surface_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "costa_like_surface_v1_grid"


# -----------------------------------------------------------------------------
# Costa-like surface geometry
#
# Artistic parametric approximation:
# central catenoid-like neck + threefold Enneper-like wings.
# -----------------------------------------------------------------------------

u = np.linspace(-1.65, 1.65, 210)
v = np.linspace(0.0, 2 * np.pi, 260)

U, V = np.meshgrid(u, v)

neck = np.cosh(0.72 * U)
threefold = 0.42 * np.cos(3 * V) * np.exp(-0.36 * U**2)

R = 0.72 * neck + threefold

X = R * np.cos(V)
Y = R * np.sin(V)
Z = 1.15 * U + 0.42 * np.sin(3 * V) * np.exp(-0.28 * U**2)

# Mild twist for a more Costa-like folded look
twist = 0.38 * U
X2 = X * np.cos(twist) - Y * np.sin(twist)
Y2 = X * np.sin(twist) + Y * np.cos(twist)

X = X2
Y = Y2

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 3.95
Y = Y / scale * 3.95
Z = Z / scale * 3.95


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.7, 3.7)
    ax.set_ylim(-3.7, 3.7)
    ax.set_zlim(-3.7, 3.7)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(X[i, :], Y[i, :], Z[i, :],
                color=COL, linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[i, :], Y[i, :], Z[i, :],
                color=COL_GRID_U, linewidth=GRID_CORE_WIDTH, alpha=core_alpha)

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(X[:, j], Y[:, j], Z[:, j],
                color=COL, linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[:, j], Y[:, j], Z[:, j],
                color=COL_GRID_V, linewidth=GRID_CORE_WIDTH, alpha=core_alpha)


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()
    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/costa_like_surface_v1_grid.webm
Frames: 192
Size: 4100.1 KB


[out#0/webm @ 0x152623a00] video:4098KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.047778%
frame=  192 fps= 27 q=32.0 Lsize=    4100KiB time=00:00:08.00 bitrate=4198.5kbits/s speed=1.11x    


# Dini Surface

In [4]:
# Dini Surface v2 — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/dini_surface_v2_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "dini_surface_v2_grid"


# -----------------------------------------------------------------------------
# Dini surface geometry
#
# x = a cos(u) sin(v)
# y = a sin(u) sin(v)
# z = a (cos(v) + log(tan(v/2))) + b u
# -----------------------------------------------------------------------------

a = 1.0
b = 0.22

u = np.linspace(0.0, 10.0 * np.pi, 280)
v = np.linspace(0.12, 1.55, 150)

U, V = np.meshgrid(u, v)

X = a * np.cos(U) * np.sin(V)
Y = a * np.sin(U) * np.sin(V)
Z = a * (np.cos(V) + np.log(np.tan(V / 2.0))) + b * U

X -= X.mean()
Y -= Y.mean()
Z -= Z.mean()

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.05
Y = Y / scale * 4.05
Z = Z / scale * 4.05


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.7, 3.7)
    ax.set_ylim(-3.7, 3.7)
    ax.set_zlim(-3.7, 3.7)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :], Y[i, :], Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=27 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/dini_surface_v2_grid.webm
Frames: 192
Size: 841.7 KB


[out#0/webm @ 0x15c0052a0] video:840KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.220126%
frame=  192 fps= 54 q=32.0 Lsize=     842KiB time=00:00:08.00 bitrate= 861.9kbits/s speed=2.24x    


$$x=a\cos u \sin v$$

$$y=a\sin u \sin v$$

$$z=a\left(
\cos v +
\ln\tan\frac{v}{2}
\right)
+b\,u$$

# Kuen Surface v2

In [5]:
# Kuen Surface v2 — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/kuen_surface_v2_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "kuen_surface_v2_grid"


u = np.linspace(-4.8, 4.8, 260)
v = np.linspace(0.08, np.pi - 0.08, 170)

U, V = np.meshgrid(u, v)

den = 1.0 + U**2 * np.sin(V) ** 2

X = (2.0 * (np.cos(U) + U * np.sin(U)) * np.sin(V)) / den
Y = (2.0 * (np.sin(U) - U * np.cos(U)) * np.sin(V)) / den
Z = np.log(np.tan(V / 2.0)) + (2.0 * np.cos(V)) / den

X -= X.mean()
Y -= Y.mean()
Z -= Z.mean()

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.15
Y = Y / scale * 4.15
Z = Z / scale * 4.15


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.75, 3.75)
    ax.set_ylim(-3.75, 3.75)
    ax.set_zlim(-3.75, 3.75)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.16 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(X[i, :], Y[i, :], Z[i, :],
                color=COL, linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[i, :], Y[i, :], Z[i, :],
                color=COL_GRID_U, linewidth=GRID_CORE_WIDTH, alpha=core_alpha)

    for j in range(0, X.shape[1], GRID_STEP_V):
        ax.plot(X[:, j], Y[:, j], Z[:, j],
                color=COL, linewidth=GRID_GLOW_WIDTH, alpha=glow_alpha)
        ax.plot(X[:, j], Y[:, j], Z[:, j],
                color=COL_GRID_V, linewidth=GRID_CORE_WIDTH, alpha=core_alpha)


frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=27 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X, Y, Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/kuen_surface_v2_grid.webm
Frames: 192
Size: 3258.1 KB


# Cross Cap

In [6]:
# Cross-Cap — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/cross_cap_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt

from vizlib.animation_export import export_animation


ANIMATION_NAME = "cross_cap_v1_grid"


u = np.linspace(0, np.pi, 220)
v = np.linspace(0, 2 * np.pi, 260)

U, V = np.meshgrid(u, v)

X = np.sin(U) * np.sin(2 * V)
Y = np.sin(2 * U) * np.cos(V) ** 2
Z = np.cos(2 * U) * np.cos(V) ** 2

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.0
Y = Y / scale * 4.0
Z = Z / scale * 4.0


fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes():
    ax.set_xlim(-3.7, 3.7)
    ax.set_ylim(-3.7, 3.7)
    ax.set_zlim(-3.7, 3.7)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color):
    hex_color = hex_color.lstrip("#")
    return tuple(
        int(hex_color[i:i + 2], 16) / 255
        for i in (0, 2, 4)
    )


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse):

    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.18 + 0.95 * zn
    brightness *= (0.74 + 0.26 * pulse)

    colors = np.zeros((*Z.shape, 4))

    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_visible_grid(pulse):

    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):

        ax.plot(
            X[i], Y[i], Z[i],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )

        ax.plot(
            X[i], Y[i], Z[i],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(0, X.shape[1], GRID_STEP_V):

        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )

        ax.plot(
            X[:, j], Y[:, j], Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


frames = []

tau = 2 * np.pi

for frame_idx in range(FRAMES):

    t = frame_idx / FRAMES

    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()

    ax.set_facecolor(BG)

    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(out_file)

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

animations/Math/cross_cap_v1_grid.webm


# Klein Quartic

In [7]:
# Klein Quartic Sculpture — rotating mathematical sculpture, museum grid style
# Output: media-site/animations/Math/klein_quartic_v1_grid.webm

import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401

from vizlib.animation_export import export_animation


ANIMATION_NAME = "klein_quartic_v1_grid"


# -----------------------------------------------------------------------------
# Klein quartic inspired geometry
#
# Seven-fold symmetry
# -----------------------------------------------------------------------------

theta = np.linspace(0.0, 2 * np.pi, 320)
phi = np.linspace(0.0, np.pi, 220)

TH, PH = np.meshgrid(theta, phi)

R = (
    1.0
    + 0.35 * np.cos(7 * TH) * np.sin(3 * PH)
    + 0.18 * np.cos(7 * PH)
)

X = R * np.sin(PH) * np.cos(TH)
Y = R * np.sin(PH) * np.sin(TH)
Z = R * np.cos(PH)

scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.1
Y = Y / scale * 4.1
Z = Z / scale * 4.1


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(
    111,
    projection="3d",
)

ax.set_facecolor(BG)


def setup_axes() -> None:

    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))

    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):

    hex_color = hex_color.lstrip("#")

    return tuple(
        int(hex_color[i:i + 2], 16) / 255
        for i in (0, 2, 4)
    )


base_rgb = np.array(
    hex_to_rgb(COL)
)


def build_colors(pulse: float):

    rn = (
        R - R.min()
    ) / (
        R.max() - R.min() + 1e-9
    )

    brightness = 0.18 + 0.95 * rn
    brightness *= (
        0.74 + 0.26 * pulse
    )

    colors = np.zeros((*R.shape, 4))

    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(
        colors,
        0,
        1,
    )


def draw_visible_grid(pulse: float):

    glow_alpha = (
        GRID_GLOW_ALPHA
        * (0.78 + 0.22 * pulse)
    )

    core_alpha = (
        GRID_CORE_ALPHA
        * (0.86 + 0.14 * pulse)
    )

    for i in range(
        0,
        X.shape[0],
        GRID_STEP_U,
    ):

        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )

        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    for j in range(
        0,
        X.shape[1],
        GRID_STEP_V,
    ):

        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )

        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []

tau = 2 * np.pi

for frame_idx in range(FRAMES):

    t = frame_idx / FRAMES

    pulse = (
        0.5
        + 0.5 * np.sin(
            tau * 4 * t
        )
    )

    ax.clear()

    ax.set_facecolor(BG)

    setup_axes()

    ax.view_init(
        elev=28 + 6 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_visible_grid(pulse)

    fig.canvas.draw()

    width, height = (
        fig.canvas.get_width_height()
    )

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(
        height,
        width,
        4,
    )

    frames.append(
        frame.copy()
    )

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/klein_quartic_v1_grid.webm
Frames: 192
Size: 4959.2 KB


[out#0/webm @ 0x136705550] video:4957KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.041055%
frame=  192 fps= 25 q=32.0 Lsize=    4959KiB time=00:00:08.00 bitrate=5078.2kbits/s speed=1.02x    


Это один из самых знаменитых объектов в геометрии.

Уравнение квартики Клейна:

$$x^3y + y^3z + z^3x = 0$$

на проективной плоскости.

В чистом виде её тяжело визуализировать, поэтому для нашей коллекции обычно строят её радиальную параметризацию. Получается нечто между:

* морским ежом;
* кристаллом;
* квантовой орбиталью;
* артефактом из Halo.

Для анимации лучше взять известную визуализационную форму:

$$r = 1 + a\cos(7\theta)\sin(3\phi) + b\cos(7\phi)$$

где число 7 не случайно — это знаменитая симметрия квартики Клейна.

# Halvorsen Attractor

In [8]:
# Halvorsen Attractor Tube — rotating mathematical sculpture
# Output: media-site/animations/Math/halvorsen_attractor_tube_v1.webm

import numpy as np
import matplotlib.pyplot as plt

from vizlib.animation_export import export_animation


ANIMATION_NAME = "halvorsen_attractor_tube_v1"


# -----------------------------------------------------------------------------
# Halvorsen attractor
#
# dx/dt = -a*x - 4*y - 4*z - y^2
# dy/dt = -a*y - 4*z - 4*x - z^2
# dz/dt = -a*z - 4*x - 4*y - x^2
# -----------------------------------------------------------------------------

A = 1.4

DT = 0.004
STEPS = 70000
SKIP = 6000
DECIMATE = 4

x = np.zeros(STEPS)
y = np.zeros(STEPS)
z = np.zeros(STEPS)

x[0] = 0.1
y[0] = 0.0
z[0] = 0.0

for i in range(STEPS - 1):
    dx = -A * x[i] - 4.0 * y[i] - 4.0 * z[i] - y[i] ** 2
    dy = -A * y[i] - 4.0 * z[i] - 4.0 * x[i] - z[i] ** 2
    dz = -A * z[i] - 4.0 * x[i] - 4.0 * y[i] - x[i] ** 2

    x[i + 1] = x[i] + DT * dx
    y[i + 1] = y[i] + DT * dy
    z[i + 1] = z[i] + DT * dz


cx = x[SKIP::DECIMATE]
cy = y[SKIP::DECIMATE]
cz = z[SKIP::DECIMATE]

cx -= cx.mean()
cy -= cy.mean()
cz -= cz.mean()

C = np.vstack([cx, cy, cz]).T


# -----------------------------------------------------------------------------
# Tube construction
# -----------------------------------------------------------------------------

n_path = len(C)
n_tube = 34

theta = np.linspace(0.0, 2.0 * np.pi, n_tube)

dC = np.gradient(C, axis=0)

tangent = dC / (np.linalg.norm(dC, axis=1, keepdims=True) + 1e-12)

up = np.array([0.0, 0.0, 1.0])
normal = np.cross(tangent, up)

bad = np.linalg.norm(normal, axis=1) < 1e-6
normal[bad] = np.cross(tangent[bad], np.array([0.0, 1.0, 0.0]))

normal = normal / (np.linalg.norm(normal, axis=1, keepdims=True) + 1e-12)

binormal = np.cross(tangent, normal)
binormal = binormal / (np.linalg.norm(binormal, axis=1, keepdims=True) + 1e-12)

tube_r = 0.045

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):
    for j in range(n_tube):
        offset = (
            tube_r * np.cos(theta[j]) * normal[i]
            + tube_r * np.sin(theta[j]) * binormal[i]
        )

        p = C[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]


scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.8
Y = Y / scale * 4.8
Z = Z / scale * 4.8


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.18 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    # Tube rings
    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    # Path-direction ribs
    rib_step = max(GRID_STEP_V * 2, 16)

    for j in range(0, X.shape[1], rib_step):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=25 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Path points: {n_path}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/halvorsen_attractor_tube_v1.webm
Path points: 16000
Frames: 192
Size: 5821.5 KB


[out#0/webm @ 0x11d622490] video:5819KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.034905%
frame=  192 fps= 24 q=32.0 Lsize=    5821KiB time=00:00:08.00 bitrate=5961.2kbits/s speed=0.989x    


Он выглядит как смесь:

* Томаса,
* Лоренца,
* спутанного клубка ДНК,
* магнитного поля.

Система:

$$\dot x = -a x - 4y - 4z - y^2$$

$$\dot y = -a y - 4z - 4x - z^2$$

$$\dot z = -a z - 4x - 4y - x^2$$

обычно:

$$a = 1.4$$

Для визуализации я бы использовал тот же стиль, что и у Aizawa:

* генерируем траекторию;
* строим трубку вокруг неё;
* полупрозрачная поверхность;
* музейная сетка.

# Dadras Attractor Tube

In [11]:
# Dadras Attractor Tube — rotating mathematical sculpture
# Output: media-site/animations/Math/dadras_attractor_tube_v1.webm

import numpy as np
import matplotlib.pyplot as plt

from vizlib.animation_export import export_animation


ANIMATION_NAME = "dadras_attractor_tube_v1"


# -----------------------------------------------------------------------------
# Dadras attractor
#
# dx/dt = y - a*x + b*y*z
# dy/dt = c*y - x*z + z
# dz/dt = d*x*y - e*z
# -----------------------------------------------------------------------------

A = 3.0
B = 2.7
C_PARAM = 1.7
D = 2.0
E = 9.0

DT = 0.002
STEPS = 90000
SKIP = 8000
DECIMATE = 5

x = np.zeros(STEPS)
y = np.zeros(STEPS)
z = np.zeros(STEPS)

x[0] = 1.0
y[0] = 1.0
z[0] = 1.0

for i in range(STEPS - 1):
    dx = y[i] - A * x[i] + B * y[i] * z[i]
    dy = C_PARAM * y[i] - x[i] * z[i] + z[i]
    dz = D * x[i] * y[i] - E * z[i]

    x[i + 1] = x[i] + DT * dx
    y[i + 1] = y[i] + DT * dy
    z[i + 1] = z[i] + DT * dz


cx = x[SKIP::DECIMATE]
cy = y[SKIP::DECIMATE]
cz = z[SKIP::DECIMATE]

cx -= cx.mean()
cy -= cy.mean()
cz -= cz.mean()

C_PATH = np.vstack([cx, cy, cz]).T


# -----------------------------------------------------------------------------
# Tube construction
# -----------------------------------------------------------------------------

n_path = len(C_PATH)
n_tube = 34

theta = np.linspace(0.0, 2.0 * np.pi, n_tube)

dC = np.gradient(C_PATH, axis=0)
tangent = dC / (np.linalg.norm(dC, axis=1, keepdims=True) + 1e-12)

up = np.array([0.0, 0.0, 1.0])
normal = np.cross(tangent, up)

bad = np.linalg.norm(normal, axis=1) < 1e-6
normal[bad] = np.cross(tangent[bad], np.array([0.0, 1.0, 0.0]))

normal = normal / (np.linalg.norm(normal, axis=1, keepdims=True) + 1e-12)

binormal = np.cross(tangent, normal)
binormal = binormal / (np.linalg.norm(binormal, axis=1, keepdims=True) + 1e-12)

tube_r = 0.045

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):
    for j in range(n_tube):
        offset = (
            tube_r * np.cos(theta[j]) * normal[i]
            + tube_r * np.sin(theta[j]) * binormal[i]
        )

        p = C_PATH[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]


scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 4.9
Y = Y / scale * 4.9
Z = Z / scale * 4.9


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.18 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    rib_step = max(GRID_STEP_V * 2, 16)

    for j in range(0, X.shape[1], rib_step):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=25 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Path points: {n_path}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/dadras_attractor_tube_v1.webm
Path points: 16400
Frames: 192
Size: 2374.1 KB


[out#0/webm @ 0x12c60b6f0] video:2372KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.079617%
frame=  192 fps= 42 q=32.0 Lsize=    2374KiB time=00:00:08.00 bitrate=2431.1kbits/s speed=1.75x    


# Rössler Attractor Tube

In [12]:
# Rössler Attractor Tube — rotating mathematical sculpture
# Output: media-site/animations/Math/rossler_attractor_tube_v1.webm

import numpy as np
import matplotlib.pyplot as plt

from vizlib.animation_export import export_animation


ANIMATION_NAME = "rossler_attractor_tube_v1"


# -----------------------------------------------------------------------------
# Rössler attractor
#
# dx/dt = -y - z
# dy/dt = x + a*y
# dz/dt = b + z*(x - c)
# -----------------------------------------------------------------------------

A = 0.2
B = 0.2
C_PARAM = 5.7

DT = 0.01
STEPS = 70000
SKIP = 6000
DECIMATE = 4

x = np.zeros(STEPS)
y = np.zeros(STEPS)
z = np.zeros(STEPS)

x[0] = 0.1
y[0] = 0.0
z[0] = 0.0

for i in range(STEPS - 1):
    dx = -y[i] - z[i]
    dy = x[i] + A * y[i]
    dz = B + z[i] * (x[i] - C_PARAM)

    x[i + 1] = x[i] + DT * dx
    y[i + 1] = y[i] + DT * dy
    z[i + 1] = z[i] + DT * dz


cx = x[SKIP::DECIMATE]
cy = y[SKIP::DECIMATE]
cz = z[SKIP::DECIMATE]

cx -= cx.mean()
cy -= cy.mean()
cz -= cz.mean()

print("Ranges:", np.ptp(cx), np.ptp(cy), np.ptp(cz))

C_PATH = np.vstack([cx, cy, cz]).T


# -----------------------------------------------------------------------------
# Tube construction
# -----------------------------------------------------------------------------

n_path = len(C_PATH)
n_tube = 34

theta = np.linspace(0.0, 2.0 * np.pi, n_tube)

dC = np.gradient(C_PATH, axis=0)
tangent = dC / (np.linalg.norm(dC, axis=1, keepdims=True) + 1e-12)

up = np.array([0.0, 0.0, 1.0])
normal = np.cross(tangent, up)

bad = np.linalg.norm(normal, axis=1) < 1e-6
normal[bad] = np.cross(tangent[bad], np.array([0.0, 1.0, 0.0]))

normal = normal / (np.linalg.norm(normal, axis=1, keepdims=True) + 1e-12)

binormal = np.cross(tangent, normal)
binormal = binormal / (np.linalg.norm(binormal, axis=1, keepdims=True) + 1e-12)

tube_r = 0.045

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):
    for j in range(n_tube):
        offset = (
            tube_r * np.cos(theta[j]) * normal[i]
            + tube_r * np.sin(theta[j]) * binormal[i]
        )

        p = C_PATH[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]


scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 5.0
Y = Y / scale * 5.0
Z = Z / scale * 5.0


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.18 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    rib_step = max(GRID_STEP_V * 2, 16)

    for j in range(0, X.shape[1], rib_step):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=28 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Path points: {n_path}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

Ranges: 21.06982739021887 19.02093581268882 26.41487884873583


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/rossler_attractor_tube_v1.webm
Path points: 16000
Frames: 192
Size: 856.2 KB


[out#0/webm @ 0x12ce1e500] video:854KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.216377%
frame=  192 fps= 66 q=32.0 Lsize=     856KiB time=00:00:08.00 bitrate= 876.8kbits/s speed=2.76x    


# Sprott Butterfly Attractor Tube

In [13]:
# Sprott Butterfly Attractor Tube — rotating mathematical sculpture
# Output: media-site/animations/Math/sprott_butterfly_attractor_tube_v1.webm

import numpy as np
import matplotlib.pyplot as plt

from vizlib.animation_export import export_animation


ANIMATION_NAME = "sprott_butterfly_attractor_tube_v1"


# -----------------------------------------------------------------------------
# Sprott Butterfly attractor
#
# dx/dt = y*z
# dy/dt = x - y
# dz/dt = 1 - x*y
# -----------------------------------------------------------------------------

DT = 0.01
STEPS = 65000
SKIP = 5000
DECIMATE = 4

x = np.zeros(STEPS)
y = np.zeros(STEPS)
z = np.zeros(STEPS)

x[0] = 0.63
y[0] = 0.47
z[0] = 0.21

for i in range(STEPS - 1):
    dx = y[i] * z[i]
    dy = x[i] - y[i]
    dz = 1.0 - x[i] * y[i]

    x[i + 1] = x[i] + DT * dx
    y[i + 1] = y[i] + DT * dy
    z[i + 1] = z[i] + DT * dz


cx = x[SKIP::DECIMATE]
cy = y[SKIP::DECIMATE]
cz = z[SKIP::DECIMATE]

cx -= cx.mean()
cy -= cy.mean()
cz -= cz.mean()

print("Ranges:", np.ptp(cx), np.ptp(cy), np.ptp(cz))

C_PATH = np.vstack([cx, cy, cz]).T


# -----------------------------------------------------------------------------
# Tube construction
# -----------------------------------------------------------------------------

n_path = len(C_PATH)
n_tube = 34

theta = np.linspace(0.0, 2.0 * np.pi, n_tube)

dC = np.gradient(C_PATH, axis=0)
tangent = dC / (np.linalg.norm(dC, axis=1, keepdims=True) + 1e-12)

up = np.array([0.0, 0.0, 1.0])
normal = np.cross(tangent, up)

bad = np.linalg.norm(normal, axis=1) < 1e-6
normal[bad] = np.cross(tangent[bad], np.array([0.0, 1.0, 0.0]))

normal = normal / (np.linalg.norm(normal, axis=1, keepdims=True) + 1e-12)

binormal = np.cross(tangent, normal)
binormal = binormal / (np.linalg.norm(binormal, axis=1, keepdims=True) + 1e-12)

tube_r = 0.042

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):
    for j in range(n_tube):
        offset = (
            tube_r * np.cos(theta[j]) * normal[i]
            + tube_r * np.sin(theta[j]) * binormal[i]
        )

        p = C_PATH[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]


scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 5.0
Y = Y / scale * 5.0
Z = Z / scale * 5.0


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.18 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    # Tube rings
    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    # Cross ribs
    rib_step = max(GRID_STEP_V * 2, 16)

    for j in range(0, X.shape[1], rib_step):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=25 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Path points: {n_path}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

Ranges: 9.195923989351368 5.713468687761489 8.227513989342688


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/sprott_butterfly_attractor_tube_v1.webm
Path points: 15000
Frames: 192
Size: 3394.1 KB


[out#0/webm @ 0x138f23640] video:3392KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.056512%
frame=  192 fps= 35 q=32.0 Lsize=    3394KiB time=00:00:08.00 bitrate=3475.5kbits/s speed=1.46x    


# Lorenz Attractor Tube

In [15]:
# Lorenz Attractor Tube — rotating mathematical sculpture
# Output: media-site/animations/Math/lorenz_attractor_tube_v1.webm

import numpy as np
import matplotlib.pyplot as plt

from vizlib.animation_export import export_animation


ANIMATION_NAME = "lorenz_attractor_tube_v1"


# -----------------------------------------------------------------------------
# Lorenz attractor
#
# dx/dt = sigma * (y - x)
# dy/dt = x * (rho - z) - y
# dz/dt = x * y - beta * z
# -----------------------------------------------------------------------------

SIGMA = 10.0
RHO = 28.0
BETA = 8.0 / 3.0

DT = 0.005
STEPS = 70000
SKIP = 5000
DECIMATE = 5

x = np.zeros(STEPS)
y = np.zeros(STEPS)
z = np.zeros(STEPS)

x[0] = 0.1
y[0] = 0.0
z[0] = 0.0

for i in range(STEPS - 1):
    dx = SIGMA * (y[i] - x[i])
    dy = x[i] * (RHO - z[i]) - y[i]
    dz = x[i] * y[i] - BETA * z[i]

    x[i + 1] = x[i] + DT * dx
    y[i + 1] = y[i] + DT * dy
    z[i + 1] = z[i] + DT * dz


cx = x[SKIP::DECIMATE]
cy = y[SKIP::DECIMATE]
cz = z[SKIP::DECIMATE]

cx -= cx.mean()
cy -= cy.mean()
cz -= cz.mean()

print("Ranges:", np.ptp(cx), np.ptp(cy), np.ptp(cz))

C_PATH = np.vstack([cx, cy, cz]).T


# -----------------------------------------------------------------------------
# Tube construction
# -----------------------------------------------------------------------------

n_path = len(C_PATH)
n_tube = 34

theta = np.linspace(0.0, 2.0 * np.pi, n_tube)

dC = np.gradient(C_PATH, axis=0)
tangent = dC / (np.linalg.norm(dC, axis=1, keepdims=True) + 1e-12)

up = np.array([0.0, 0.0, 1.0])
normal = np.cross(tangent, up)

bad = np.linalg.norm(normal, axis=1) < 1e-6
normal[bad] = np.cross(tangent[bad], np.array([0.0, 1.0, 0.0]))

normal = normal / (np.linalg.norm(normal, axis=1, keepdims=True) + 1e-12)

binormal = np.cross(tangent, normal)
binormal = binormal / (np.linalg.norm(binormal, axis=1, keepdims=True) + 1e-12)

tube_r = 0.04

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):
    for j in range(n_tube):
        offset = (
            tube_r * np.cos(theta[j]) * normal[i]
            + tube_r * np.sin(theta[j]) * binormal[i]
        )

        p = C_PATH[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]


scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 5.0
Y = Y / scale * 5.0
Z = Z / scale * 5.0


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.18 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    rib_step = max(GRID_STEP_V * 2, 16)

    for j in range(0, X.shape[1], rib_step):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    t = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * t)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=25 + 5 * np.sin(tau * t),
        azim=360 * t,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Path points: {n_path}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

Ranges: 38.47820595634395 52.08818498851548 46.865264262222155


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/lorenz_attractor_tube_v1.webm
Path points: 13000
Frames: 192
Size: 4485.5 KB


[out#0/webm @ 0x13f620cf0] video:4484KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.043409%
frame=  192 fps= 29 q=32.0 Lsize=    4486KiB time=00:00:08.00 bitrate=4593.2kbits/s speed=1.21x    


Для классического аттрактора Лоренца система имеет вид:

$$ \begin{aligned}\frac{dx}{dt}&=\sigma(y-x)\\\frac{dy}{dt}&=x(\rho-z)-y\\\frac{dz}{dt}&=xy-\beta z\end{aligned} $$

Наиболее часто используются параметры:

$$ \sigma = 10,\qquad $$
$$ \rho = 28,\qquad $$
$$ \beta = \frac{8}{3} $$

При этих значениях возникает знаменитая «бабочка Лоренца» — хаотический аттрактор с двумя лопастями, между которыми траектория непредсказуемо переключается.

# Figure-8 Knot Tube 

In [16]:
# Figure-8 Knot Tube — rotating mathematical sculpture
# Output: media-site/animations/Math/figure8_knot_tube_v1.webm

import numpy as np
import matplotlib.pyplot as plt

from vizlib.animation_export import export_animation


ANIMATION_NAME = "figure8_knot_tube_v1"


# -----------------------------------------------------------------------------
# Figure-8 Knot
#
# x = (2 + cos(2t)) cos(3t)
# y = (2 + cos(2t)) sin(3t)
# z = sin(4t)
# -----------------------------------------------------------------------------

N_POINTS = 14000

t = np.linspace(0.0, 2.0 * np.pi, N_POINTS)

cx = (2.0 + np.cos(2.0 * t)) * np.cos(3.0 * t)
cy = (2.0 + np.cos(2.0 * t)) * np.sin(3.0 * t)
cz = np.sin(4.0 * t)

cz *= 1.55

cx -= cx.mean()
cy -= cy.mean()
cz -= cz.mean()

print("Ranges:", np.ptp(cx), np.ptp(cy), np.ptp(cz))

C_PATH = np.vstack([cx, cy, cz]).T


# -----------------------------------------------------------------------------
# Tube construction
# -----------------------------------------------------------------------------

n_path = len(C_PATH)
n_tube = 42

theta = np.linspace(0.0, 2.0 * np.pi, n_tube)

dC = np.gradient(C_PATH, axis=0)
tangent = dC / (np.linalg.norm(dC, axis=1, keepdims=True) + 1e-12)

up = np.array([0.0, 0.0, 1.0])
normal = np.cross(tangent, up)

bad = np.linalg.norm(normal, axis=1) < 1e-6
normal[bad] = np.cross(tangent[bad], np.array([0.0, 1.0, 0.0]))

normal = normal / (np.linalg.norm(normal, axis=1, keepdims=True) + 1e-12)

binormal = np.cross(tangent, normal)
binormal = binormal / (np.linalg.norm(binormal, axis=1, keepdims=True) + 1e-12)

tube_r = 0.055

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):
    for j in range(n_tube):
        offset = (
            tube_r * np.cos(theta[j]) * normal[i]
            + tube_r * np.sin(theta[j]) * binormal[i]
        )

        p = C_PATH[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]


scale = np.max(np.abs([X, Y, Z]))

X = X / scale * 5.4
Y = Y / scale * 5.4
Z = Z / scale * 5.4


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.18 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


def draw_grid(pulse: float) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    rib_step = max(GRID_STEP_V * 2, 16)

    for j in range(0, X.shape[1], rib_step):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2 * np.pi

for frame_idx in range(FRAMES):
    tt = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4 * tt)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=26 + 5 * np.sin(tau * tt),
        azim=360 * tt,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        rstride=1,
        cstride=1,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_grid(pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Path points: {n_path}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

Ranges: 5.999999219384108 5.117301163713586 3.0999999804846015


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/figure8_knot_tube_v1.webm
Path points: 14000
Frames: 192
Size: 2205.3 KB


[out#0/webm @ 0x143f10ca0] video:2203KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.085449%
frame=  192 fps= 47 q=32.0 Lsize=    2205KiB time=00:00:08.00 bitrate=2258.3kbits/s speed=1.97x    


#Cinquefoil Knot Tube

In [17]:
# Cinquefoil Knot Tube — rotating mathematical sculpture
# Output: media-site/animations/Math/cinquefoil_knot_tube_v1.webm

import numpy as np
import matplotlib.pyplot as plt

from vizlib.animation_export import export_animation


ANIMATION_NAME = "cinquefoil_knot_tube_v1"


# -----------------------------------------------------------------------------
# Cinquefoil Knot (5₁)
#
# Torus knot (p=2, q=5)
# -----------------------------------------------------------------------------

N_POINTS = 16000

t = np.linspace(
    0.0,
    2.0 * np.pi,
    N_POINTS,
)

R = 2.4
r = 0.95

cx = (
    R + r * np.cos(5 * t)
) * np.cos(2 * t)

cy = (
    R + r * np.cos(5 * t)
) * np.sin(2 * t)

cz = (
    r * np.sin(5 * t)
)

cz *= 1.35

cx -= cx.mean()
cy -= cy.mean()
cz -= cz.mean()

print(
    "Ranges:",
    np.ptp(cx),
    np.ptp(cy),
    np.ptp(cz),
)

C_PATH = np.vstack(
    [cx, cy, cz]
).T


# -----------------------------------------------------------------------------
# Tube construction
# -----------------------------------------------------------------------------

n_path = len(C_PATH)
n_tube = 44

theta = np.linspace(
    0.0,
    2.0 * np.pi,
    n_tube,
)

dC = np.gradient(
    C_PATH,
    axis=0,
)

tangent = dC / (
    np.linalg.norm(
        dC,
        axis=1,
        keepdims=True,
    )
    + 1e-12
)

up = np.array([0.0, 0.0, 1.0])

normal = np.cross(
    tangent,
    up,
)

bad = (
    np.linalg.norm(
        normal,
        axis=1,
    ) < 1e-6
)

normal[bad] = np.cross(
    tangent[bad],
    np.array([0.0, 1.0, 0.0]),
)

normal = normal / (
    np.linalg.norm(
        normal,
        axis=1,
        keepdims=True,
    )
    + 1e-12
)

binormal = np.cross(
    tangent,
    normal,
)

binormal = binormal / (
    np.linalg.norm(
        binormal,
        axis=1,
        keepdims=True,
    )
    + 1e-12
)

tube_r = 0.06

X = np.zeros((n_tube, n_path))
Y = np.zeros((n_tube, n_path))
Z = np.zeros((n_tube, n_path))

for i in range(n_path):

    for j in range(n_tube):

        offset = (
            tube_r * np.cos(theta[j]) * normal[i]
            + tube_r * np.sin(theta[j]) * binormal[i]
        )

        p = C_PATH[i] + offset

        X[j, i] = p[0]
        Y[j, i] = p[1]
        Z[j, i] = p[2]


scale = np.max(
    np.abs([X, Y, Z])
)

X = X / scale * 5.6
Y = Y / scale * 5.6
Z = Z / scale * 5.6


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(
    figsize=(W, H),
    dpi=DPI,
)

fig.patch.set_facecolor(BG)

ax = fig.add_subplot(
    111,
    projection="3d",
)

ax.set_facecolor(BG)


def setup_axes():

    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))

    ax.grid(False)
    ax.set_axis_off()


def hex_to_rgb(hex_color):

    hex_color = hex_color.lstrip("#")

    return tuple(
        int(hex_color[i:i+2], 16)/255
        for i in (0, 2, 4)
    )


base_rgb = np.array(
    hex_to_rgb(COL)
)


def build_colors(pulse):

    zn = (
        Z - Z.min()
    ) / (
        Z.max() - Z.min() + 1e-9
    )

    brightness = (
        0.18
        + 0.95 * zn
    )

    brightness *= (
        0.74
        + 0.26 * pulse
    )

    colors = np.zeros((*Z.shape, 4))

    colors[...,0] = base_rgb[0] * brightness
    colors[...,1] = base_rgb[1] * brightness
    colors[...,2] = base_rgb[2] * brightness
    colors[...,3] = SURFACE_ALPHA

    return np.clip(
        colors,
        0,
        1,
    )


def draw_grid(pulse):

    glow_alpha = (
        GRID_GLOW_ALPHA
        * (0.78 + 0.22 * pulse)
    )

    core_alpha = (
        GRID_CORE_ALPHA
        * (0.86 + 0.14 * pulse)
    )

    for i in range(
        0,
        X.shape[0],
        GRID_STEP_U,
    ):

        ax.plot(
            X[i],
            Y[i],
            Z[i],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )

        ax.plot(
            X[i],
            Y[i],
            Z[i],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    rib_step = max(
        GRID_STEP_V * 2,
        16,
    )

    for j in range(
        0,
        X.shape[1],
        rib_step,
    ):

        ax.plot(
            X[:,j],
            Y[:,j],
            Z[:,j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )

        ax.plot(
            X[:,j],
            Y[:,j],
            Z[:,j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []

tau = 2*np.pi

for frame_idx in range(FRAMES):

    tt = frame_idx / FRAMES

    pulse = (
        0.5
        + 0.5*np.sin(
            tau * 4 * tt
        )
    )

    ax.clear()

    ax.set_facecolor(BG)

    setup_axes()

    ax.view_init(
        elev=26 + 5*np.sin(tau*tt),
        azim=360*tt,
    )

    ax.plot_surface(
        X,
        Y,
        Z,
        facecolors=build_colors(pulse),
        linewidth=0,
        antialiased=True,
        shade=False,
        alpha=SURFACE_ALPHA,
    )

    draw_grid(pulse)

    fig.canvas.draw()

    width, height = (
        fig.canvas.get_width_height()
    )

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(
        height,
        width,
        4,
    )

    frames.append(
        frame.copy()
    )

plt.close(fig)

out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(out_file)

Ranges: 6.297079909910746 6.491123036696964 2.5649999876373535


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

animations/Math/cinquefoil_knot_tube_v1.webm


[out#0/webm @ 0x13e625700] video:2067KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.091120%
frame=  192 fps= 53 q=32.0 Lsize=    2069KiB time=00:00:08.00 bitrate=2118.9kbits/s speed=2.22x    


# Borromean Rings

In [18]:
# Borromean Rings — rotating mathematical sculpture
# Output: media-site/animations/Math/borromean_rings_v1.webm

import numpy as np
import matplotlib.pyplot as plt

from vizlib.animation_export import export_animation


ANIMATION_NAME = "borromean_rings_v1"


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------

def make_tube(
    C_PATH: np.ndarray,
    tube_r: float = 0.055,
    n_tube: int = 42,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:

    n_path = len(C_PATH)
    theta = np.linspace(0.0, 2.0 * np.pi, n_tube)

    dC = np.gradient(C_PATH, axis=0)
    tangent = dC / (np.linalg.norm(dC, axis=1, keepdims=True) + 1e-12)

    up = np.array([0.0, 0.0, 1.0])
    normal = np.cross(tangent, up)

    bad = np.linalg.norm(normal, axis=1) < 1e-6
    normal[bad] = np.cross(tangent[bad], np.array([0.0, 1.0, 0.0]))

    normal = normal / (np.linalg.norm(normal, axis=1, keepdims=True) + 1e-12)

    binormal = np.cross(tangent, normal)
    binormal = binormal / (np.linalg.norm(binormal, axis=1, keepdims=True) + 1e-12)

    X = np.zeros((n_tube, n_path))
    Y = np.zeros((n_tube, n_path))
    Z = np.zeros((n_tube, n_path))

    for i in range(n_path):
        for j in range(n_tube):
            offset = (
                tube_r * np.cos(theta[j]) * normal[i]
                + tube_r * np.sin(theta[j]) * binormal[i]
            )

            p = C_PATH[i] + offset

            X[j, i] = p[0]
            Y[j, i] = p[1]
            Z[j, i] = p[2]

    return X, Y, Z


def hex_to_rgb(hex_color: str):
    hex_color = hex_color.lstrip("#")
    return tuple(int(hex_color[i:i + 2], 16) / 255 for i in (0, 2, 4))


base_rgb = np.array(hex_to_rgb(COL))


def build_colors(Z: np.ndarray, pulse: float):
    zn = (Z - Z.min()) / (Z.max() - Z.min() + 1e-9)

    brightness = 0.18 + 0.95 * zn
    brightness *= 0.74 + 0.26 * pulse

    colors = np.zeros((*Z.shape, 4))
    colors[..., 0] = base_rgb[0] * brightness
    colors[..., 1] = base_rgb[1] * brightness
    colors[..., 2] = base_rgb[2] * brightness
    colors[..., 3] = SURFACE_ALPHA

    return np.clip(colors, 0, 1)


# -----------------------------------------------------------------------------
# Borromean-like rings geometry
# -----------------------------------------------------------------------------

N_POINTS = 5000

t = np.linspace(0.0, 2.0 * np.pi, N_POINTS)

R = 1.65
OFFSET = 0.72

# Ring 1 — mostly XY plane
x1 = R * np.cos(t) + OFFSET
y1 = R * np.sin(t)
z1 = 0.38 * np.sin(2.0 * t)

# Ring 2 — mostly YZ plane
x2 = 0.38 * np.sin(2.0 * t + 2.0 * np.pi / 3.0)
y2 = R * np.cos(t) + OFFSET
z2 = R * np.sin(t)

# Ring 3 — mostly ZX plane
x3 = R * np.cos(t)
y3 = 0.38 * np.sin(2.0 * t + 4.0 * np.pi / 3.0)
z3 = R * np.sin(t) + OFFSET

rings_paths = [
    np.vstack([x1, y1, z1]).T,
    np.vstack([x2, y2, z2]).T,
    np.vstack([x3, y3, z3]).T,
]

# Center whole system
all_points = np.vstack(rings_paths)
center = all_points.mean(axis=0)

rings_paths = [
    C_PATH - center
    for C_PATH in rings_paths
]


# -----------------------------------------------------------------------------
# Tubes
# -----------------------------------------------------------------------------

rings = [
    make_tube(C_PATH, tube_r=0.06, n_tube=42)
    for C_PATH in rings_paths
]

all_values = np.concatenate([
    arr.reshape(-1)
    for ring in rings
    for arr in ring
])

scale = np.max(np.abs(all_values))

scaled_rings = []

for X, Y, Z in rings:
    scaled_rings.append((
        X / scale * 5.2,
        Y / scale * 5.2,
        Z / scale * 5.2,
    ))


# -----------------------------------------------------------------------------
# Figure
# -----------------------------------------------------------------------------

fig = plt.figure(figsize=(W, H), dpi=DPI)
fig.patch.set_facecolor(BG)

ax = fig.add_subplot(111, projection="3d")
ax.set_facecolor(BG)


def setup_axes() -> None:
    ax.set_xlim(-3.8, 3.8)
    ax.set_ylim(-3.8, 3.8)
    ax.set_zlim(-3.8, 3.8)

    ax.set_box_aspect((1, 1, 1))
    ax.grid(False)
    ax.set_axis_off()


def draw_grid(
    X: np.ndarray,
    Y: np.ndarray,
    Z: np.ndarray,
    pulse: float,
) -> None:
    glow_alpha = GRID_GLOW_ALPHA * (0.78 + 0.22 * pulse)
    core_alpha = GRID_CORE_ALPHA * (0.86 + 0.14 * pulse)

    for i in range(0, X.shape[0], GRID_STEP_U):
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[i, :],
            Y[i, :],
            Z[i, :],
            color=COL_GRID_U,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )

    rib_step = max(GRID_STEP_V * 2, 16)

    for j in range(0, X.shape[1], rib_step):
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL,
            linewidth=GRID_GLOW_WIDTH,
            alpha=glow_alpha,
        )
        ax.plot(
            X[:, j],
            Y[:, j],
            Z[:, j],
            color=COL_GRID_V,
            linewidth=GRID_CORE_WIDTH,
            alpha=core_alpha,
        )


# -----------------------------------------------------------------------------
# Frames
# -----------------------------------------------------------------------------

frames = []
tau = 2.0 * np.pi

for frame_idx in range(FRAMES):
    tt = frame_idx / FRAMES
    pulse = 0.5 + 0.5 * np.sin(tau * 4.0 * tt)

    ax.clear()
    ax.set_facecolor(BG)
    setup_axes()

    ax.view_init(
        elev=27 + 5 * np.sin(tau * tt),
        azim=360 * tt,
    )

    for X, Y, Z in scaled_rings:
        ax.plot_surface(
            X,
            Y,
            Z,
            rstride=1,
            cstride=1,
            facecolors=build_colors(Z, pulse),
            linewidth=0,
            antialiased=True,
            shade=False,
            alpha=SURFACE_ALPHA,
        )

        draw_grid(X, Y, Z, pulse)

    fig.canvas.draw()

    width, height = fig.canvas.get_width_height()

    frame = np.frombuffer(
        fig.canvas.buffer_rgba(),
        dtype=np.uint8,
    ).reshape(height, width, 4)

    frames.append(frame.copy())

plt.close(fig)


out_file = export_animation(
    frames=frames,
    out_dir=OUT_DIR,
    animation_name=ANIMATION_NAME,
    output_format=OUTPUT_FORMAT,
    fps=FPS,
    alpha=ALPHA,
)

print(f"Saved: {out_file}")
print(f"Rings: {len(scaled_rings)}")
print(f"Frames: {len(frames)}")
print(f"Size: {out_file.stat().st_size / 1024:.1f} KB")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with clang version 18.1.8
  configuration: --prefix=/Users/mloktionov/anaconda3/envs/astro-ai --cc=arm64-apple-darwin20.0.0-clang --cxx=arm64-apple-darwin20.0.0-clang++ --nm=arm64-apple-darwin20.0.0-nm --ar=arm64-apple-darwin20.0.0-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libopenh264 --enable-libdav1d --enable-cross-compile --arch=arm64 --target-os=darwin --cross-prefix=arm64-apple-darwin20.0.0- --host-cc=/Users/runner/miniforge3/conda-bld/ffmpeg_1748704173249/_build_env/bin/x86_64-apple-darwin13.4.0-clang --enable-neon --disable-gnutls --enable-libvpx --enable-libass --enable-pthreads --enable-libopenvino --enable-gpl --enable-libx264 --enable-libx265 --enable-libmp3lame --enable-libaom --enable-libsvtav1 --enable-libxml2 --enable-pic --enable-shared --disable-static --enable-version3 --enable-zlib

Saved: animations/Math/borromean_rings_v1.webm
Rings: 3
Frames: 192
Size: 2978.0 KB


[out#0/webm @ 0x156e204a0] video:2976KiB audio:0KiB subtitle:0KiB other streams:0KiB global headers:0KiB muxing overhead: 0.063922%
frame=  192 fps= 42 q=32.0 Lsize=    2978KiB time=00:00:08.00 bitrate=3049.4kbits/s speed=1.74x    
